## CLIP zero-shot moderation - ViT-L/14


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torchvision import transforms
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, TaskType, get_peft_model

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "notebooks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'openai/clip-vit-large-patch14'
device = "cuda" if torch.cuda.is_available() else "cpu"
sns.set_theme(style="whitegrid")


In [ ]:
def load_image_frame() -> pd.DataFrame:
    for candidate in [Path(path) for path in DATASETS]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.suffix == ".parquet":
            frame = pd.read_parquet(candidate)
        else:
            frame = pd.read_csv(candidate)
        break
    else:
        raise FileNotFoundError("No local image manifest was found.")

    if "image_path" not in frame.columns:
        for column in frame.columns:
            if "path" in column.lower():
                frame = frame.rename(columns={column: "image_path"})
                break
    if "label" not in frame.columns:
        label_like = [col for col in frame.columns if "label" in col.lower() or "nsfw" in col.lower()]
        frame["label"] = frame[label_like[0]] if label_like else np.where(frame.index % 2 == 0, 0, 1)
    frame["label"] = pd.to_numeric(frame["label"], errors="coerce").fillna(0).astype(int)
    frame = frame[frame["image_path"].map(lambda value: Path(str(value)).exists())].copy()
    return frame[["image_path", "label"]].reset_index(drop=True)


image_df = load_image_frame()
train_df, valid_df = train_test_split(
    image_df,
    test_size=0.2 if len(image_df) >= 50 else 0.3,
    stratify=image_df["label"] if image_df["label"].nunique() > 1 else None,
    random_state=42,
)
clip_model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }

def compute_metrics_with_prompts(images: list[Image.Image], labels: list[int], prompts: list[str], threshold: float = 0.5) -> dict:
    prompt_sets = {
        "plain": ["safe for work image", "not safe for work image"],
        "platform_policy": ["acceptable social media content", "explicit adult content"],
        "strict": ["benign neutral photo", "graphic sexual content"],
    }
    augmentation = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
        ]
    )


    @torch.inference_mode()
    def score_with_prompts(images: list[Image.Image], prompts: list[str]) -> np.ndarray:
        inputs = clip_processor(text=prompts, images=images, return_tensors="pt", padding=True).to(device)
        outputs = clip_model(**inputs)
        logits = outputs.logits_per_image.detach().cpu().numpy()
        shifted = logits - logits.max(axis=1, keepdims=True)
        probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
        return probs[:, 1]


    rows = []
    for _, row in valid_df.head(min(6, len(valid_df))).iterrows():
        image = Image.open(row["image_path"]).convert("RGB")
        variants = {"original": image, "augmented": augmentation(image)}
        for prompt_name, prompts in prompt_sets.items():
            for variant_name, variant_image in variants.items():
                rows.append(
                    {
                        "path": row["image_path"],
                        "label": row["label"],
                        "prompt_set": prompt_name,
                        "variant": variant_name,
                        "score": float(score_with_prompts([variant_image], prompts)[0]),
                    }
                )
    analysis_df = pd.DataFrame(rows)
    display(analysis_df)


In [ ]:
batch_images = [Image.open(path).convert("RGB") for path in valid_df["image_path"].head(min(8, len(valid_df)))]
batch_prompts = ["safe for work image", "not safe for work image"]
inputs = clip_processor(text=batch_prompts, images=batch_images, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    outputs = clip_model(**inputs, output_hidden_states=True)

image_embeds = outputs.image_embeds.detach().cpu().numpy()
text_embeds = outputs.text_embeds.detach().cpu().numpy()
similarities = image_embeds @ text_embeds.T

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(similarities, cmap="mako", ax=axes[0])
sns.heatmap(pd.DataFrame(image_embeds).corr(), cmap="crest", ax=axes[1])
plt.tight_layout()


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


def embed_images(paths: list[str], batch_size: int = 8) -> np.ndarray:
    features = []
    for start in range(0, len(paths), batch_size):
        images = [Image.open(path).convert("RGB") for path in paths[start : start + batch_size]]
        inputs = clip_processor(images=images, return_tensors="pt").to(device)
        with torch.no_grad():
            vectors = clip_model.get_image_features(**inputs).detach().cpu().numpy()
        features.append(vectors)
    return np.vstack(features)


train_features = embed_images(train_df["image_path"].tolist())
valid_features = embed_images(valid_df["image_path"].tolist())
linear_probe = LogisticRegression(max_iter=3000, class_weight="balanced")
linear_probe.fit(train_features, train_df["label"])
valid_scores = linear_probe.predict_proba(valid_features)[:, 1]
base_metrics = compute_binary_metrics(valid_df["label"], valid_scores, threshold=0.5)

probe_dir = ARTIFACT_ROOT / "clip_linear_probe"
probe_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(train_features).to_parquet(probe_dir / "train_embeddings.parquet", index=False)
pd.DataFrame(valid_features).to_parquet(probe_dir / "valid_embeddings.parquet", index=False)
(probe_dir / "metrics.json").write_text(json.dumps(base_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
base_metrics


In [ ]:
lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    use_dora=False,
)
dora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    use_dora=True,
)

clip_lora = get_peft_model(CLIPModel.from_pretrained(MODEL_ID), lora_cfg)
clip_dora = get_peft_model(CLIPModel.from_pretrained(MODEL_ID), dora_cfg)
clip_lora.save_pretrained(ARTIFACT_ROOT / "clip_lora_adapter")
clip_dora.save_pretrained(ARTIFACT_ROOT / "clip_dora_adapter")


In [ ]:
#Temp